# Ingest

Reports the data-row count of every England extract made by `python -m plotluck.sources extract` (see `docs/server.md`). It reads the extracts only: pulls run in the terminal. Counts only, never rows (ADR 0003). On full data the counts go to `runs/<yyyy-mm-dd>-ingest/` to commit; on sample data, to the sample's `run/` folder.

In [ ]:
import json
from datetime import date

from plotluck import data_dir, sources

data = data_dir()
full = data == data_dir(choice="full")
run = data.parents[1] / "runs" / f"{date.today():%Y-%m-%d}-ingest" if full else data / "run"
print(data, "->", run)

In [ ]:
# Stops with an error naming the source if an extract is missing or has no data rows.
counts = sources.row_counts(data)
print(f"{'source':<17} {'data rows':>10}  retrieved")
for name, rows in counts.items():
    prov = data / "raw" / f"{name}.provenance.json"
    retrieved = json.loads(prov.read_text(encoding="utf-8"))["retrieved_at"][:10] if prov.is_file() else "-"
    print(f"{name:<17} {rows:>10,}  {retrieved}")

In [ ]:
run.mkdir(parents=True, exist_ok=True)
(run / "row_counts.json").write_text(json.dumps(counts, indent=2) + "\n", encoding="utf-8", newline="\n")
print("wrote", run / "row_counts.json")